#  ⭐ `dim_atc`

**Objetivo**  
Construir a dimensão `dim_atc` na camada **Gold** a partir de dados **Silver**, normalizando códigos ATC e princípios ativos (WhoDrug), aplicando regras de mapeamento para gerar `PK_ATC` (código ATC final) e `DESCRICAO` (princípio ativo padronizado), e entregando um conjunto limpo, consistente e auditável para consumo analítico (dashboards e modelos).

**Escopo**
- **Entradas (🥈 Silver):**
  - Tabela/lista de medicamentos com colunas mínimas:  
    `CODIGO_ATC`, `PRINCIPIOS_ATIVOS_WHODRUG`, `NOME_MEDICAMENTO_WHODRUG`, (opcionalmente `DOSE`, `FREQUENCIA_DOSE`, `INICIO_ADMINISTRACAO`, `FIM_ADMINISTRACAO`).
  - **Regras de mapeamento** (`regras`): `{"atc", "principio", "pk", "descricao"}`.
- **Transformações principais:**
  1) Normalização (maiúsculas, remoção de acentos).  
  2) Aplicação das regras para preencher `PK_ATC` e `DESCRICAO`.  
  3) Limpeza e padronização de posologia (dose, frequência, datas).
  4) _Joins_ de validação quando necessário (ex.: com catálogo de medicamentos).
  5) Checks de qualidade e geração de métricas.
- **Saída (🥇Gold):**
  - `dim_atc` (parquet/csv) com colunas padronizadas:  
    `PK_ATC`, `DESCRICAO`, `CODIGO_ATC`, `PRINCIPIOS_ATIVOS_WHODRUG`, colunas de auditoria (`etl_run_id`, `data_processamento`), e campos opcionais de posologia normalizada (`DOSE_NORM`, `FREQUENCIA_DOSE_H`, `INICIO_ADMINISTRACAO_DT`, `FIM_ADMINISTRACAO_DT`).

**Granularidade & Chave**
- Granularidade: nível **ATC x princípio ativo** (um registro por par mapeado relevante).  
- Chave lógica: `PK_ATC`. Em casos de homônimos, use `PK_ATC` + `DESCRICAO`.

**Usuários & Consumo**
- Dashboards clínico-farmacológicos, análises de farmacovigilância, e _features_ para modelos (ex.: sinalização por classes ATC).

**Assunções**
- `regras` cobre os principais princípios ativos de interesse.
- Entradas já passaram por validação básica (tipos e nulos críticos).

**Rastreabilidade**
- Cada execução registra `etl_run_id`, horário de processamento e métricas de cobertura das regras.


In [1]:
%run ../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_principio_ativo_atc

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
dim_medicamentos = pd.read_parquet(path) 
dim_medicamentos = dim_medicamentos[['CODIGO_ATC', 'PRINCIPIOS_ATIVOS_WHODRUG', 'NOME_MEDICAMENTO_WHODRUG',
                                    # 'DETENTOR_REGISTRO', 'DOSE','VIA_ADMINISTRACAO'
                                     ]].drop_duplicates().sort_values(by=['CODIGO_ATC', 'PRINCIPIOS_ATIVOS_WHODRUG', 'NOME_MEDICAMENTO_WHODRUG']) 
dim_medicamentos['PK_MEDICAMENTO'] = np.arange(len(dim_medicamentos))
dim_medicamentos = dim_medicamentos.reset_index(drop=True)
dim_medicamentos = dim_medicamentos[['PK_MEDICAMENTO', 'CODIGO_ATC',  'PRINCIPIOS_ATIVOS_WHODRUG', 'NOME_MEDICAMENTO_WHODRUG']]
dim_medicamentos.head()

,PK_MEDICAMENTO,CODIGO_ATC,PRINCIPIOS_ATIVOS_WHODRUG,NOME_MEDICAMENTO_WHODRUG
0,0,A,None,Alimentary tract and metabolism
1,1,A01A,Mucin,Saliva
2,2,A01A,Sodium chlorate,Sodium chlorate
3,3,A01AA,Fluoreto de sódio,Dual
4,4,A01AA,Flúor_x000D_|Xilitol,Biotene


### Quality

In [4]:
dim_medicamentos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20225 entries, 0 to 20224
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   PK_MEDICAMENTO             20225 non-null  int64 
 1   CODIGO_ATC                 20224 non-null  object
 2   PRINCIPIOS_ATIVOS_WHODRUG  20124 non-null  object
 3   NOME_MEDICAMENTO_WHODRUG   20224 non-null  object
dtypes: int64(1), object(3)
memory usage: 632.2+ KB


In [5]:
dim_medicamentos.CODIGO_ATC.isnull().sum()

np.int64(1)

In [6]:
dim_medicamentos.PRINCIPIOS_ATIVOS_WHODRUG.isnull().sum()

np.int64(101)

In [7]:
dim_medicamentos.NOME_MEDICAMENTO_WHODRUG.isnull().sum()

np.int64(1)

# 🥈 Silver

normalized data, medium quality


## hist_atc

In [8]:
# essa etapa passar para data\02_silver\hist_atc\rules.txt
regras = [
    {"atc": "L04AB", "principio": "adalimumab",         "pk": "L04AB04", "descricao": "ADALIMUMAB"},
    {"atc": "L04AB", "principio": "certolizumab pegol", "pk": "L04AB05", "descricao": "CERTOLIZUMAB PEGOL"},
    {"atc": "L04AB", "principio": "etanercept",         "pk": "L04AB01", "descricao": "ETANERCEPT"},
    {"atc": "L04AB", "principio": "golimumab",          "pk": "L04AB06", "descricao": "GOLIMUMAB"},
    {"atc": "L04AB", "principio": "infliximab",         "pk": "L04AB02", "descricao": "INFLIXIMAB"},
    {"atc": "L04AA", "principio": "abatacept",          "pk": "L04AA24", "descricao": "ABATACEPT"},
    {"atc": "L01FA", "principio": "rituximab",          "pk": "L01FA01", "descricao": "RITUXIMAB"},
    {"atc": "L04AC", "principio": "tocilizumab",        "pk": "L04AC07", "descricao": "TOCILIZUMAB"}
]


In [9]:
hst_atc = normalize_principio_ativo_atc(
    dim_medicamentos,
    pk_atc_col="PK_ATC",
    descricao_col="DESCRICAO",
    regras=regras
)

In [10]:
hst_atc.query("PK_ATC=='L04AA24'").head()

,PK_MEDICAMENTO,CODIGO_ATC,PRINCIPIOS_ATIVOS_WHODRUG,NOME_MEDICAMENTO_WHODRUG,PK_ATC,DESCRICAO
14172,14172,L04AA,Abatacept,Abatacept,L04AA24,ABATACEPT
14173,14173,L04AA,Abatacept,Abataceptum,L04AA24,ABATACEPT
14174,14174,L04AA,Abatacept,Orencia,L04AA24,ABATACEPT


In [11]:
hst_atc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20225 entries, 0 to 20224
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   PK_MEDICAMENTO             20225 non-null  int64 
 1   CODIGO_ATC                 20224 non-null  object
 2   PRINCIPIOS_ATIVOS_WHODRUG  20124 non-null  object
 3   NOME_MEDICAMENTO_WHODRUG   20224 non-null  object
 4   PK_ATC                     20225 non-null  object
 5   DESCRICAO                  20225 non-null  object
dtypes: int64(1), object(5)
memory usage: 948.2+ KB


In [15]:
hst_atc.to_parquet(Path(project_root) / "data/02_silver/hist_atc/hist_atc.parquet", index=False)

# 🥇 Gold



## dim_atc

In [14]:
dim_atc = hst_atc[['PK_ATC', 'DESCRICAO']].drop_duplicates()
dim_atc.to_parquet(Path(project_root) / "data/03_gold/dim_atc/dim_atc.parquet", index=False)